In [ ]:
!pip install jax "jax[cuda13]" transformers huggingface_hub einops

### Gemma 3

This notebook walks you through implementing the core of the Gemma 3 transformer model from scratch using JAX. 

Gemma 3 is a family of open-weights, lightweight, and multimodal AI models from Google. Gemma 3 handles both text and image inputs to generate text, allowing for sophisticated visual understanding. Below you will find code for the Gemma 3 Text Model only. 

Gemma 3 Technical Report: https://arxiv.org/pdf/2503.19786.   
HF Source: https://huggingface.co/google/gemma-3-4B-it

### Imports

In [1]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

In [ ]:
import glob
import json
import os
from dataclasses import dataclass
from functools import partial
from typing import Optional

import numpy as np
from PIL import Image
import jax
import jax.numpy as jnp
from safetensors.numpy import load_file
from transformers import AutoTokenizer

In [ ]:
import os
os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxx" # Your HF Token here

In [4]:
HF_REPO_ID = "google/gemma-3-4b-it"
LOCAL_DIR_PATH = "gemma-3-4b-it"

The below code defines the configuration for the Gemma 3 4B IT model. This allows us to easily manage and access the model's hyperparameters throughout our implementation. We use the 4B parameter configuration as a base, so that you can run the model on minimal hardware. You can adjust these parameters to match the larger configurations (3B, 7B, 30B) when needed.

In [5]:
@dataclass(frozen=True)
class TextConfig:
    hidden_size: int = 2560
    intermediate_size: int = 10240
    num_hidden_layers: int = 34
    sliding_window: int = 1024
    rope_scaling_factor: float = 8.0
    rope_scaling_type: str = "linear"

@dataclass(frozen=True)
class VisionConfig:
    hidden_size: int = 1152
    image_size: int = 896
    intermediate_size: int = 4304
    num_attention_heads: int = 16
    num_hidden_layers: int = 27
    patch_size: int = 14

@dataclass(frozen=True)
class Gemma3Config:
    # Multimodal
    boi_token_index: int = 255999
    eoi_token_index: int = 256000
    eos_token_id: tuple = (1, 106)
    image_token_index: int = 262144
    mm_tokens_per_image: int = 256
    text_config: TextConfig = TextConfig()
    vision_config: VisionConfig = VisionConfig()
    
    # Text model
    hidden_size: int = 2560
    num_layers: int = 34
    num_heads: int = 8
    num_kv_heads: int = 4
    head_dim: int = 256
    intermediate_size: int = 10240
    # max_seq_len: int = 32768
    max_seq_len: int = 8192
    rms_norm_eps: float = 1e-6
    query_pre_attn_scalar: int = 256
    sliding_window: int = 1024
    sliding_window_pattern: int = 6
    vocab_size: int = 262144

config = Gemma3Config()

In this section, we download the weights of the model of interest from huggingface to use in our implementation.

In [ ]:
from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id=HF_REPO_ID,
    local_dir=LOCAL_DIR_PATH,
)

print(f"Downloaded repository path: {local_dir}")

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Downloaded repository path: /workspace/gemma-3-4b-it


In [6]:
tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR_PATH)
print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


### Weights Load

In [7]:
shard_paths = sorted(glob.glob(os.path.join(LOCAL_DIR_PATH, "*.safetensors")))

if not shard_paths:
    raise FileNotFoundError(f"No safetensors found in {LOCAL_DIR_PATH}")

combined_weights = {}
for path in shard_paths:
    print(f"Loading shard: {os.path.basename(path)}...")
    shard = load_file(path)
    combined_weights.update(shard)
        
hf_weights = combined_weights

Loading shard: model-00001-of-00002.safetensors...
Loading shard: model-00002-of-00002.safetensors...


In [ ]:
def get_w(
    name: str,
    transpose: bool = False,
):
    val = hf_weights.pop(name)
    val = val.T if transpose else val
    w = jnp.array(val, dtype=jnp.bfloat16)
    return w

language_model_layers = []
for i in range(config.text_config.num_hidden_layers):
    layer = {
        'attn': {
            'q_proj': get_w(f'language_model.model.layers.{i}.self_attn.q_proj.weight', True),
            'k_proj': get_w(f'language_model.model.layers.{i}.self_attn.k_proj.weight', True),
            'v_proj': get_w(f'language_model.model.layers.{i}.self_attn.v_proj.weight', True),
            'o_proj': get_w(f'language_model.model.layers.{i}.self_attn.o_proj.weight', True),
            'q_norm': {'scale': get_w(f'language_model.model.layers.{i}.self_attn.q_norm.weight')},
            'k_norm': {'scale': get_w(f'language_model.model.layers.{i}.self_attn.k_norm.weight')},
        },
        'mlp': {
            'gate_proj': get_w(f'language_model.model.layers.{i}.mlp.gate_proj.weight', True),
            'up_proj':   get_w(f'language_model.model.layers.{i}.mlp.up_proj.weight', True),
            'down_proj': get_w(f'language_model.model.layers.{i}.mlp.down_proj.weight', True),
        },
        'input_norm': {
            'scale': get_w(f'language_model.model.layers.{i}.input_layernorm.weight')
        },
        'post_attn_norm': {
            'scale': get_w(f'language_model.model.layers.{i}.post_attention_layernorm.weight')
        },
        'pre_ff_norm': {
            'scale': get_w(f'language_model.model.layers.{i}.pre_feedforward_layernorm.weight')
        },  
        'post_ff_norm': {
            'scale': get_w(f'language_model.model.layers.{i}.post_feedforward_layernorm.weight')
        },
        'is_sliding': (i not in [5, 11, 17, 23, 29, 35])
    }
    language_model_layers.append(layer)

cache = [{
            'k': jnp.zeros((config.max_seq_len, config.num_kv_heads, config.head_dim)),
            'v': jnp.zeros((config.max_seq_len, config.num_kv_heads, config.head_dim)),
        } for _ in range(config.text_config.num_hidden_layers)]

vision_model_layers = []
for i in range(config.vision_config.num_hidden_layers):
    layer = {
        'attn': {
            'k_proj': {
                'weight': get_w(f'vision_tower.vision_model.encoder.layers.{i}.self_attn.k_proj.weight', True),
                'bias': get_w(f'vision_tower.vision_model.encoder.layers.{i}.self_attn.k_proj.bias'),
            },
            'q_proj': {
                'weight': get_w(f'vision_tower.vision_model.encoder.layers.{i}.self_attn.q_proj.weight', True),
                'bias': get_w(f'vision_tower.vision_model.encoder.layers.{i}.self_attn.q_proj.bias'),
            },
            'v_proj': {
                'weight': get_w(f'vision_tower.vision_model.encoder.layers.{i}.self_attn.v_proj.weight', True),
                'bias': get_w(f'vision_tower.vision_model.encoder.layers.{i}.self_attn.v_proj.bias'),
            },
            'o_proj': {
                'weight': get_w(f'vision_tower.vision_model.encoder.layers.{i}.self_attn.out_proj.weight', True),
                'bias': get_w(f'vision_tower.vision_model.encoder.layers.{i}.self_attn.out_proj.bias'),
            },
        },
        'mlp': {
            'fc1': {
                'weight': get_w(f'vision_tower.vision_model.encoder.layers.{i}.mlp.fc1.weight', True),
                'bias': get_w(f'vision_tower.vision_model.encoder.layers.{i}.mlp.fc1.bias'),
            },
            'fc2': {
                'weight': get_w(f'vision_tower.vision_model.encoder.layers.{i}.mlp.fc2.weight', True),
                'bias': get_w(f'vision_tower.vision_model.encoder.layers.{i}.mlp.fc2.bias'),
            },
        },
        'layer_norm': {
            'layer_norm1': {
                'scale': get_w(f'vision_tower.vision_model.encoder.layers.{i}.layer_norm1.weight'),
                'bias': get_w(f'vision_tower.vision_model.encoder.layers.{i}.layer_norm1.bias'),
            },
            'layer_norm2': {
                'scale': get_w(f'vision_tower.vision_model.encoder.layers.{i}.layer_norm2.weight'),
                'bias': get_w(f'vision_tower.vision_model.encoder.layers.{i}.layer_norm2.bias'),
            },
        },
    }
    vision_model_layers.append(layer)

m = {
    'language_model': {
        'embed': get_w('language_model.model.embed_tokens.weight'),
        'layers': language_model_layers,
        'final_norm': get_w('language_model.model.norm.weight'),
    },
    'vision_model': {
        'embedding': {
            'patch': {
                'scale': get_w("vision_tower.vision_model.embeddings.patch_embedding.weight"),
                'bias': get_w("vision_tower.vision_model.embeddings.patch_embedding.bias"),
            },
            'position': get_w("vision_tower.vision_model.embeddings.position_embedding.weight"),
        },
        'layers': vision_model_layers,
        'final_norm': {
            'bias': get_w('vision_tower.vision_model.post_layernorm.bias'),
            'scale': get_w('vision_tower.vision_model.post_layernorm.weight')
        }
    },
    'mm_projector': {
        'proj': get_w('multi_modal_projector.mm_input_projection_weight'),
        'norm': get_w('multi_modal_projector.mm_soft_emb_norm.weight')
    }
}

print("Successfully Loaded Weights")

Successfully Loaded Weights


In [9]:
hf_weights.keys()

dict_keys([])

### Gemma 3 Text:

- 5:1 Interleaved Attention Pattern: Local Layers: 5 out of every 6 layers use a Sliding Window Attention (SWA) with a fixed span of 1024 tokens. Global Layers: Only every 6th layer is a "Global" layer that attends to the full 128k context.

- Frequency-Split RoPE (Rotary Embeddings): Global Layers: Use a base frequency of 1,000,000 (1M). This high frequency allows the model to distinguish between tokens that are very far apart in a 32K sequence. Local Layers: Use a base frequency of 10,000 (10k). Since they only ever look at 512 tokens, they don't need the high-frequency resolution of the global layers.

### Cache implementation

Without a cache, if you want to predict the 101st word in a sentence, the model has to re-calculate the math $K$ and $V$ for the previous 100 words every single time. This means the work grows quadratically ($O(N^2)$). 

During the Prefill phase (processing your initial prompt), the model calculates $K$ and $V$ for all your tokens and saves them in the cache. During the Decode phase (generating new tokens), the model only calculates $Q, K, V$ for the new token. It then takes the new Query and compares it against all the Keys stored in the cache to decide which Values are important

It turns a $O(N^2)$ problem into a $O(N)$ problem.

In [63]:
cache = [{
            'k': jnp.zeros((config.max_seq_len, config.num_kv_heads, config.head_dim), dtype=jnp.bfloat16),
            'v': jnp.zeros((config.max_seq_len, config.num_kv_heads, config.head_dim), dtype=jnp.bfloat16),
        } for _ in range(config.num_layers)]

RMSNorm is a simplified version of LayerNorm. While standard LayerNorm re-centers a vector (subtracts the mean) and then re-scales it (divides by variance), RMSNorm only re-scales.

$$\text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}$$

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma$$
Where $\gamma$ is a learnable scaling parameter and $\epsilon$ is a tiny constant for numerical stability.

In [11]:
def rms_norm(
        x: jnp.ndarray,        # (..., d)
        scale: jnp.ndarray,    # (d,)
        eps: float = 1e-6
    ) -> jnp.ndarray:
    ms = jnp.mean(jnp.square(x), axis=-1, keepdims=True)
    normed = x * jax.lax.rsqrt(ms + eps)
    return normed * (1.0 + scale)


Rotary Positional Embeddings (RoPE) is the piece of math that allows the model to understand the order of words without needing a fixed map of positions. 

Older models (like GPT-2) just added a fixed vector to the word, which acted like a GPS coordinate ($x + \text{pos}$). RoPE is different. Instead of adding a number, it rotates the vectors in a high-dimensional circle.

The math of RoPE is beautiful because when the model compares two words (the dot product), the result only depends on the angle between them (their relative distance), not their absolute position in the sentence. Most important is Sementic Preservation where rotating a vector doesn't change its length or magnitude, only its direction.

In [12]:
def gemma_rotary_embedding(
        position_ids: jnp.ndarray,     # (seq_len,)
        head_dim: int,
        theta: float = 1000000.0,
        attention_scaling: float = 1.0
    ) -> tuple[jnp.ndarray, jnp.ndarray]:
    inv_freq = 1.0 / (theta ** (jnp.arange(0, head_dim, 2) / head_dim))
    t = jnp.arange(position_ids.max() + 1)
    freqs = jnp.outer(t, inv_freq)
    emb = jnp.concatenate([freqs, freqs], axis=-1)
    cos = (jnp.cos(emb[position_ids]) * attention_scaling).astype(jnp.bfloat16)
    sin = (jnp.sin(emb[position_ids]) * attention_scaling).astype(jnp.bfloat16)
    return cos, sin


In [13]:
def apply_rope(
        x: jnp.ndarray,    # (seq_len, num_heads, head_dim)
        cos: jnp.ndarray,  # (seq_len, head_dim)
        sin: jnp.ndarray   # (seq_len, head_dim)
    ) -> jnp.ndarray:
    cos = cos[:, None, :]
    sin = sin[:, None, :]
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    rotate_half_x = jnp.concatenate([-x2, x1], axis=-1)
    return x * cos + rotate_half_x * sin


In JAX, we implement this by creating a precomputed "rotation matrix" and applying it to your $Q$ and $K$ vectors before the attention score calculation.

In [14]:
rotary_embeddings_table = {
    'global': gemma_rotary_embedding(jnp.arange(config.max_seq_len), config.head_dim, theta=1000000.0),
    'local': gemma_rotary_embedding(jnp.arange(config.max_seq_len), config.head_dim, theta=10000.0)
}

In [15]:
def gemma_rotary_embedding_lookup(
        position_ids: jnp.ndarray,                          # (seq_len,)
        embeddings_table: tuple[jnp.ndarray, jnp.ndarray]   # each (max_seq_len, head_dim)
    ) -> tuple[jnp.ndarray, jnp.ndarray]:
    cos_table, sin_table = embeddings_table
    cos = cos_table[position_ids]
    sin = sin_table[position_ids]
    return cos, sin


Here we define a function for updating the key and value caches during autoregressive generation. This function takes the current cache, the new key and value tensors, and the starting position in the sequence. It uses JAX's dynamic_update_slice to efficiently update the cache without needing to copy the entire tensor.

In [16]:
def update_cache(
        cache_dict: dict[str, jnp.ndarray],
        new_k: jnp.ndarray,    # (seq_len, num_kv_heads, head_dim)
        new_v: jnp.ndarray,    # (seq_len, num_kv_heads, head_dim)
        start_pos: int
    ) -> tuple[jnp.ndarray, jnp.ndarray, dict[str, jnp.ndarray]]:
    seq_len = new_k.shape[0]
    end_pos = start_pos + seq_len
    updated_k = jax.lax.dynamic_update_slice(cache_dict['k'], new_k, (start_pos, 0, 0))
    updated_v = jax.lax.dynamic_update_slice(cache_dict['v'], new_v, (start_pos, 0, 0))
    updated_cache = {'k': updated_k, 'v': updated_v}
    return updated_k, updated_v, updated_cache


The padding mask is used to indicate which tokens in the input sequence are actual data and which are padding. During the prefill stage, we have a fixed input sequence length, and any positions beyond the actual input tokens are padded with zeros. The padding mask helps the model distinguish between real tokens and padding, ensuring that the attention mechanism only focuses on the relevant tokens in the sequence.

This is important for the attention mechanism, as it allows the model to ignore the padded positions when computing attention scores. The function takes in the input token IDs and the maximum sequence length, and returns a mask that can be applied to the attention scores to prevent attending to padded positions.

In [17]:
def get_padding_mask(
        input_ids: jnp.ndarray,    # (seq_len,)
        max_seq_len: int
    ) -> jnp.ndarray:              # (1, seq_len, max_seq_len)
    padding_mask = (input_ids > 0).astype(jnp.bfloat16)
    mask_expanded = padding_mask[jnp.newaxis, :, jnp.newaxis]
    final_row_mask = jnp.broadcast_to(mask_expanded, (1, len(input_ids), max_seq_len))
    return final_row_mask


The causal mask is used to ensure that each token can only attend to previous tokens in the sequence. This is crucial for autoregressive models like Gemma 3, where the prediction of the next token should not have access to future tokens. The function supports both sliding window attention and full attention, depending on the configuration of the layer.

If sliding window attention is enabled, the mask will only allow attention to a fixed number of previous tokens (defined by window_size). If full attention is used, the mask will allow attention to all previous tokens in the sequence. This flexibility allows us to optimize memory usage and computational efficiency based on the specific requirements of each layer in the model.

In [18]:
@partial(jax.jit, static_argnums=(0,1,2,3,4))
def get_causal_mask(
    seq_len: int,
    start_pos: int,
    sliding: bool,
    window_size: int,
    max_seq_len: int
    ) -> jnp.ndarray:              # (1, seq_len, max_seq_len)
    
    q_idx = jnp.arange(start_pos, start_pos + seq_len)[:, None]
    k_idx = jnp.arange(max_seq_len)[None, :]

    if sliding:
        causal_mask = ((q_idx >= k_idx) & (q_idx - window_size < k_idx)).astype(jnp.bfloat16)
        causal_mask = causal_mask[None, :, :]
    else:
        causal_mask = (q_idx >= k_idx).astype(jnp.bfloat16)
        causal_mask = causal_mask[None, :, :]

    return causal_mask


In [19]:
def gemma_attention_block(
        inputs_ids: jnp.ndarray,                                # (seq_len,)
        hidden_states: jnp.ndarray,                             # (seq_len, hidden_size)
        layer: dict[str, jnp.ndarray],
        position_embeddings: tuple[jnp.ndarray, jnp.ndarray],   # each (seq_len, head_dim)
        is_sliding: bool,
        start_pos: int,
        cache: dict[str, jnp.ndarray],
        causal_mask: jnp.ndarray = None
    ) -> tuple[jnp.ndarray, dict[str, jnp.ndarray]]:

    seq_len, _ = hidden_states.shape
    end_pos = start_pos + seq_len

    q = jnp.dot(hidden_states, layer['q_proj'])
    k = jnp.dot(hidden_states, layer['k_proj'])
    v = jnp.dot(hidden_states, layer['v_proj'])

    q = q.reshape(seq_len, config.num_heads, config.head_dim)
    k = k.reshape(seq_len, config.num_kv_heads, config.head_dim)
    v = v.reshape(seq_len, config.num_kv_heads, config.head_dim)

    q = rms_norm(q, layer['q_norm']['scale'])
    k = rms_norm(k, layer['k_norm']['scale'])

    cos, sin = position_embeddings
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)

    k, v, updated_cache = update_cache(cache, k, v, start_pos)

    k = jnp.repeat(k, config.num_heads // config.num_kv_heads, axis=1)
    v = jnp.repeat(v, config.num_heads // config.num_kv_heads, axis=1)

    q, k, v = q.transpose(1, 0, 2), k.transpose(1, 0, 2), v.transpose(1, 0, 2)

    logits = jnp.matmul(q, k.transpose(0, 2, 1)) / jnp.sqrt(config.query_pre_attn_scalar)

    padding_mask = get_padding_mask(inputs_ids, config.max_seq_len)

    if causal_mask is None:
        causal_mask = get_causal_mask(seq_len, start_pos, is_sliding, config.sliding_window, config.max_seq_len)

    logits = logits + (1.0 - causal_mask) * -1e10

    weights = jax.nn.softmax(logits, axis=-1)

    weights = weights * padding_mask

    output = jnp.matmul(weights, v)
    output = output.transpose(1, 0, 2).reshape(seq_len, -1)

    output = jnp.dot(output, layer['o_proj'])

    return output, updated_cache


Feedforward network (MLP) implementation. This is a simple two-layer feedforward network with a GELU activation in between. The "gate_proj" and "up_proj" are the two linear transformations, and the output is projected back to the hidden size using "down_proj". The gating mechanism allows for more expressive power in the MLP, as it can modulate the activations based on the input.

In [20]:
def gemma_mlp(
        x: jnp.ndarray,              # (seq_len, hidden_size)
        params: dict[str, jnp.ndarray]
    ) -> jnp.ndarray:
    gate = jnp.dot(x, params['gate_proj'])
    up   = jnp.dot(x, params['up_proj'])
    activated = jax.nn.gelu(gate, approximate=True) * up
    return jnp.dot(activated, params['down_proj'])


In [21]:
def gemma_decoder_block(
        inputs_ids: jnp.ndarray,                              # (seq_len,)
        hidden_states: jnp.ndarray,                           # (seq_len, hidden_size)
        layer: dict[str, jnp.ndarray],
        position_embeddings: tuple[jnp.ndarray, jnp.ndarray],# each (seq_len, head_dim)
        start_pos: int,
        cache: dict[str, jnp.ndarray],
        causal_mask: jnp.ndarray = None
    ) -> tuple[jnp.ndarray, dict[str, jnp.ndarray]]:
    
    residual_connections = hidden_states
    hidden_states = rms_norm(hidden_states, layer['input_norm']['scale'])

    hidden_states, updated_cache = gemma_attention_block(inputs_ids, hidden_states, layer['attn'], position_embeddings, layer['is_sliding'], start_pos, cache, causal_mask)
    hidden_states = rms_norm(hidden_states, layer['post_attn_norm']['scale']) + residual_connections

    residual_connections = hidden_states
    hidden_states = rms_norm(hidden_states, layer['pre_ff_norm']['scale'])
    hidden_states = gemma_mlp(hidden_states, layer['mlp'])
    hidden_states = rms_norm(hidden_states, layer['post_ff_norm']['scale']) + residual_connections

    return hidden_states, updated_cache


In [22]:
def gemma_text_model_prefill(
        inputs_ids: jnp.ndarray,                     # (seq_len,)
        m: dict[str, jnp.ndarray],
        cache: list[dict[str, jnp.ndarray]],
        rotary_embeddings_table: dict[str, jnp.ndarray],
        config: Gemma3Config,
        start_pos: int = 0,
        hidden_states: jnp.ndarray = None,
        causal_mask: jnp.ndarray = None
    ) -> tuple[jnp.ndarray, list[dict[str, jnp.ndarray]]]:

    if hidden_states is None:
        input_embeddings = m['embed'][inputs_ids] * jnp.sqrt(config.hidden_size)
        hidden_states = input_embeddings
    
    seq_len = hidden_states.shape[0]

    position_ids = jnp.arange(seq_len) + start_pos

    pos_emb_local = gemma_rotary_embedding_lookup(position_ids, rotary_embeddings_table['local'])
    pos_emb_global = gemma_rotary_embedding_lookup(position_ids, rotary_embeddings_table['global'])

    updated_cache = []
    for layer_idx, layer in enumerate(m['layers']):
        cos = jax.lax.select(layer['is_sliding'], pos_emb_local[0], pos_emb_global[0])
        sin = jax.lax.select(layer['is_sliding'], pos_emb_local[1], pos_emb_global[1])
        position_embeddings = (cos, sin)

        layer_output, layer_cache = gemma_decoder_block(inputs_ids, hidden_states, layer, position_embeddings, start_pos, cache[layer_idx], causal_mask)
        hidden_states = layer_output
        updated_cache.append(layer_cache)

    hidden_states = rms_norm(hidden_states, m['final_norm'])
    logits = jnp.dot(hidden_states, m['embed'].T)
    return logits, updated_cache


The Generate (or Decode) function is the "creative" stage. It switches to an autoregressive mode, meaning it predicts exactly one token at a time, adds it to the sentence, and repeats. It performs a smaller matrix-vector multiplication. Instead of looking at the whole prompt again, it simply "glances" at the KV Cache to remember the past.

This phase determines the Inter-Token Latency (ITL) or "tokens per second."

In [23]:
def gemma_text_model_generate(
        inputs_ids: jnp.ndarray,                     # (1,)
        m: dict[str, jnp.ndarray],
        cache: list[dict[str, jnp.ndarray]],
        rotary_embeddings_table: dict[str, jnp.ndarray],
        config: Gemma3Config,
        start_pos: int = 0
    ) -> tuple[jnp.ndarray, list[dict[str, jnp.ndarray]]]:
    input_embeddings = m['embed'][inputs_ids] * jnp.sqrt(config.hidden_size)
    hidden_states = input_embeddings

    seq_len = hidden_states.shape[0]
    position_ids = jnp.arange(seq_len) + start_pos

    pos_emb_local = gemma_rotary_embedding_lookup(position_ids, rotary_embeddings_table['local'])
    pos_emb_global = gemma_rotary_embedding_lookup(position_ids, rotary_embeddings_table['global'])

    updated_cache = []
    for layer_idx, layer in enumerate(m['layers']):
        cos = jax.lax.select(layer['is_sliding'], pos_emb_local[0], pos_emb_global[0])
        sin = jax.lax.select(layer['is_sliding'], pos_emb_local[1], pos_emb_global[1])
        position_embeddings = (cos, sin)

        layer_output, layer_cache = gemma_decoder_block(inputs_ids, hidden_states, layer, position_embeddings, start_pos, cache[layer_idx])
        hidden_states = layer_output
        updated_cache.append(layer_cache)

    hidden_states = rms_norm(hidden_states, m['final_norm'])
    logits = jnp.dot(hidden_states, m['embed'].T)
    return logits, updated_cache


This function takes the raw logits output from the model and samples a token index based on the specified temperature. A lower temperature will make the sampling more deterministic (favoring higher probability tokens), while a higher temperature will make it more random. 

The function uses JAX's random categorical sampling to select a token index according to the probabilities derived from the logits.

In [24]:
def sample_token(
        logits: jnp.ndarray,    # (vocab_size,)
        temperature: float = 1.0,
        key: jax.random.PRNGKey = None
    ) -> jnp.ndarray:
    if key is None:
        key = jax.random.PRNGKey(0)
    scaled_logits = logits / temperature
    probs = jax.nn.softmax(scaled_logits)
    token = jax.random.categorical(key, jnp.log(probs))
    return token


The prepare_input function below takes a raw text prompt, tokenizes it, and prepares it for input into the model. It converts the prompt into token IDs, pads them to a fixed length (MAX_PROMPT_LEN), and returns both the padded token IDs and the actual length of the original token sequence. This is important because the model needs fixed-size inputs, but we also want to keep track of how much of that input is real data versus padding.

In [25]:
MAX_PROMPT_LEN = config.max_seq_len

In [26]:
def prepare_input(
        prompt: str
    ) -> tuple[jnp.ndarray, int]:  # (MAX_PROMPT_LEN,), actual_len
    tokens = tokenizer(prompt, return_tensors="np")['input_ids'][0]
    actual_len = len(tokens)
    padded_tokens = jnp.pad(tokens, (0, MAX_PROMPT_LEN - actual_len))
    return jnp.array(padded_tokens), actual_len


In [60]:
prompt = "Who is Gemma?"
input_ids, actual_len = prepare_input(prompt)
start_pos = 0
key = jax.random.PRNGKey(42)

In [29]:
# The first call to "gemma_text_model_prefill" processes the entire prompt and fills the cache with the model's "memory" of that prompt. The logits returned from this function are used to sample the next token after the prompt, which is then fed into "gemma_text_model_generate" in a loop to generate subsequent tokens one at a time.

print("Generating: ")

logits, cache = gemma_text_model_prefill(input_ids, m['language_model'], cache, rotary_embeddings_table, config, start_pos)
next_token = sample_token(logits[actual_len - 1], temperature=0.2, key=key)
all_tokens = list(input_ids[:actual_len]) + [int(next_token)]

print(prompt)
print(tokenizer.decode([int(next_token)]))

for step in range(30):
    start_pos = len(all_tokens) - 1
    logits, cache = gemma_text_model_generate(jnp.array([next_token]), m['language_model'], cache, rotary_embeddings_table, config, start_pos)
    next_token = sample_token(logits[-1], temperature=0.3, key=key)
    all_tokens.append(int(next_token))
    print(tokenizer.decode([int(next_token)]), end="", flush=True)

Generating: 
Who is Gemma?



Gemma is a large language model created by the Gemma team at Google DeepMind. She is an open-weights model, meaning that her weights are

⚠️ Note on Production Performance

The iterative loop used in this notebook is designed for readability and debugging. In a real-world production environment (like Google’s Gemini or Gemma services), we would not use a standard Python for loop for two reasons:

- Python Overhead: Each iteration of a Python loop incurs dispatch overhead where the CPU tells the GPU what to do. This creates tiny gaps in execution that add up to massive latency.
- Compilation Optimization: JAX cannot optimize across Python loop boundaries.

To make this production-ready, we would wrap the entire decoding sequence (e.g., 100+ tokens) into a single compiled block:

- jax.lax.scan: We replace the for loop with jax.lax.scan. This tells the XLA compiler, "Here is a loop of 100 steps; please compile it into one single optimized kernel on the GPU."
- Static Shapes: Production code uses fixed-size KV caches. Instead of letting the cache grow dynamically (which triggers slow recompilations), we pre-allocate a "buffer" (e.g., 2048 tokens) and use indices to update it.
- Zero-Copy Updates: By using jax.lax.dynamic_update_slice, we modify the KV cache in place (conceptually) without moving data back and forth between the CPU and GPU.

## Gemma 3 Image

In order to have multimodal capabilities of Gemma 3, we require a system that can digest multiple images simultaneously, interleave them with a text prompt, and reason across all inputs in a single forward pass.

Gemma 3 treats images as a sequence of 256 soft tokens that live in the same high-dimensional space as your text. We will handle this by encoding each image independently and then "stitching" them into the final Transformer sequence.

Unlike older models that use "ImageNet" constants, SigLIP maps the pixel range from [0, 255] to [-1, 1].

$$\text{Normalized} = \frac{(\text{Pixel} / 255) - 0.5}{0.5}$$

In [29]:
def preprocess_image(
        image_path: str,
        target_size: tuple = (896, 896)
    ) -> jnp.ndarray:              # (896, 896, 3)
    img = Image.open(image_path).convert("RGB")
    img = img.resize(target_size, resample=Image.BICUBIC)
    pixel_array = jnp.array(img)
    pixels = pixel_array
    pixels = pixels / 255.0
    mean = jnp.array([0.5, 0.5, 0.5])
    std = jnp.array([0.5, 0.5, 0.5])
    pixels = (pixels - mean) / std
    return pixels.astype(jnp.bfloat16)


### The Processing Strategy
To keep the tutorial clean, we will follow a three-step pipeline:

- Vision Encoding: Pass both images through our siglip_transformer to get two sets of tokens.
- Multimodal Interleaving: Place the image tokens into the specific slots defined by the <image> tags in your prompt.
- Cross-Attention: Use the Block-Diagonal Mask to ensure the model can see "across" both images while maintaining text causality.

Gemma 3 uses a 14x14 patch size.
- Input Image: $896 \times 896$ pixels.
- Patch Size: $14 \times 14$ pixels.
- Grid: $896 / 14 = 64$ patches along the height and $64$ along the width.
- Total Tokens: $64 \times 64 = 4,096$ patches.

### Vision Tower

This is where raw pixels are transformed into the "soft tokens" that the Transformer can understand.

The Vision Tower in Gemma 3 is built on a custom SigLIP (Sigmoid Language-Image Pre-training) encoder, followed by a Multi-Modal (MM) Projector.

### The SigLIP Vision Encoder

SigLIP is a Vision Transformer (ViT) that specializes in understanding spatial semantics. Unlike the original CLIP, which uses a contrastive loss, SigLIP uses a sigmoid loss, making it much better at localized feature extraction, essential for describing specific details in an image.

Patchify: The (896, 896) image is chopped into (14, 14) pixel patches. 

In [31]:
def siglip_patch_embedding(
        pixel_values: jnp.ndarray,  # (1, 896, 896, 3)
        params: dict
    ) -> jnp.ndarray:               # (1, 4096, hidden_dim)
    kernel_weights = params['scale']
    bias = params['bias']
    kernel_jax = kernel_weights.transpose(2, 3, 1, 0)
    
    x = jax.lax.conv_general_dilated(
        pixel_values, 
        kernel_jax, 
        window_strides=(14, 14), 
        padding='VALID',
        dimension_numbers=('NHWC', 'HWIO', 'NHWC')
    )
    x = x + bias 
    batch_size = x.shape[0]
    return x.reshape(batch_size, -1, x.shape[-1])


Linear Projection: These patches are projected into the SigLIP hidden dimension.

Transformer Backbone: The tokens pass through a deep stack of Transformer layers (27 layers). Each layer uses LayerNorm and Self-Attention so every patch can "talk" to every other patch.

Unlike the causal (one-way) attention used in the language model, SigLIP attention is fully bidirectional. Every patch in the image can "look" at every other patch simultaneously.

The Workflow:
- Linear Projection (with Bias): We project our hidden states into Query ($Q$), Key ($K$), and Value ($V$) vectors. Unlike the Gemma text layers, SigLIP almost always includes a bias term in these projections to capture subtle spatial priors.
- Scaling: We apply a scaling factor of $1/\sqrt{d_{head}}$ to the Queries. This prevents the dot-product values from growing too large, which would cause the gradients to vanish during training.
- The Score Matrix: We compute the dot product of $Q$ and $K^T$. This results in a "map" of which parts of the image are most relevant to each other (e.g., a patch containing a "whisker" attending heavily to a patch containing an "ear").
- Softmax & Context: We apply a softmax to convert these scores into probabilities (attention weights) and use them to weight the $V$ vectors.
- Output Projection: Finally, we project the merged attention heads back into the model's main dimension ($1152$) and add a final bias.

In [32]:
def siglip_attention(
        hidden_states: jnp.ndarray,  # (B, L, D)
        params: dict,
        num_heads: int
    ) -> jnp.ndarray:                # (B, L, D)
    B, L, D = hidden_states.shape
    head_dim = D // num_heads
    
    q = jnp.dot(hidden_states, params['q_proj']['weight']) + params['q_proj']['bias']
    k = jnp.dot(hidden_states, params['k_proj']['weight']) + params['k_proj']['bias']
    v = jnp.dot(hidden_states, params['v_proj']['weight']) + params['v_proj']['bias']
    
    q = q.reshape(B, L, num_heads, head_dim).transpose(0, 2, 1, 3)
    k = k.reshape(B, L, num_heads, head_dim).transpose(0, 2, 1, 3)
    v = v.reshape(B, L, num_heads, head_dim).transpose(0, 2, 1, 3)
    
    q = q * (head_dim**-0.5)
    
    attn_weights = jax.nn.softmax(jnp.matmul(q, k.transpose(0, 1, 3, 2)), axis=-1)
    attn_output = jnp.matmul(attn_weights, v)
    attn_output = attn_output.transpose(0, 2, 1, 3).reshape(B, L, D)
    
    hidden_states = jnp.dot(attn_output, params['o_proj']['weight']) + params['o_proj']['bias']
    return hidden_states

def layer_norm(
        x: jnp.ndarray,  # (..., D)
        params: dict,
        eps: float = 1e-6
    ) -> jnp.ndarray:    # (..., D)
    mean = jnp.mean(x, axis=-1, keepdims=True)
    var = jnp.var(x, axis=-1, keepdims=True)
    x = (x - mean) * jax.lax.rsqrt(var + eps)
    return x * params['scale'] + params['bias']


SigLIP's MLP is a standard two-layer feedforward network: a linear projection (`fc1`) followed by exact GELU activation, then a second projection (`fc2`) back to the hidden dimension. Unlike the Gemma text MLP, there is no gating mechanism

In [33]:
def siglip_mlp(
        x: jnp.ndarray,  # (B, L, D)
        params: dict
    ) -> jnp.ndarray:    # (B, L, D)
    x = jnp.dot(x, params['fc1']['weight']) + params['fc1']['bias']
    x = jax.nn.gelu(x, approximate=False) 
    x = jnp.dot(x, params['fc2']['weight']) + params['fc2']['bias']
    return x


In [34]:
def siglip_encoder_layer(
        hidden_states: jnp.ndarray,  # (B, L, D)
        params: dict,
        num_heads: int
    ) -> jnp.ndarray:                # (B, L, D)
    residual = hidden_states
    hidden_states = layer_norm(hidden_states, params['layer_norm']['layer_norm1'])
    hidden_states = siglip_attention(hidden_states, params['attn'], num_heads=num_heads)
    hidden_states = residual + hidden_states

    residual = hidden_states
    hidden_states = layer_norm(hidden_states, params['layer_norm']['layer_norm2'])
    hidden_states = siglip_mlp(hidden_states, params['mlp'])
    hidden_states = residual + hidden_states

    return hidden_states


The encoder consists of multiple layers of attention and MLP blocks. Each layer takes the output from the previous layer, applies layer normalization, computes attention, adds a residual connection, then applies another layer normalization followed by an MLP and another residual connection. The final output is a set of image embeddings that have been processed through these layers, ready to be fused with the text embeddings in the cross-modal layers.


In [35]:
def siglip_encoder(
        image_embeddings: jnp.ndarray,  # (1, 4096, D)
        params: list[dict],
        num_layers: int = 1,
        num_heads: int = 16
    ) -> jnp.ndarray:                   # (1, 4096, D)
    hidden_states = image_embeddings
    for i in range(num_layers):
        hidden_states = siglip_encoder_layer(hidden_states, params[i], num_heads=num_heads)
    return hidden_states


Token Condensation (C-Abstractor): The resulting $64 \times 64$ grid ($4096$ tokens) is mean-pooled into a compact $16 \times 16$ grid, giving us the final 256 vision tokens.

In [36]:
def token_condensation(
        hidden_states: jnp.ndarray  # (B, 4096, D)
    ) -> jnp.ndarray:               # (B, 256, D)
    B, L, D = hidden_states.shape
    grid_size = int(jnp.sqrt(L))
    x = hidden_states.reshape(B, grid_size, grid_size, D)
    x = x.reshape(B, grid_size // 4, 4, grid_size // 4, 4, D)
    x = jnp.mean(x, axis=(2, 4))
    return x.reshape(B, -1, D)


In [67]:
def siglip_transformer(
        image: jnp.ndarray,  # (896, 896, 3)
        params: dict
    ) -> jnp.ndarray:        # (1, 256, D)
    patch_embeds = siglip_patch_embedding(image[None, :], params['embedding']['patch'])
    embeddings = patch_embeds + params['embedding']['position'][None, :, :]
    encoded = siglip_encoder(embeddings, params['layers'], num_layers=config.vision_config.num_hidden_layers, num_heads=config.vision_config.num_attention_heads)
    last_hidden_state = layer_norm(encoded, params['final_norm'])
    vision_tokens = token_condensation(last_hidden_state)
    return vision_tokens


Now that we have our image processing pipeline defined, we can take our two standardized images, pass them through the SigLip transformer, and get back a set of vision tokens for each image. 

These tokens are then concatenated together to form a single sequence of vision embeddings that can be fed into the multi-modal projector and then into the language model for cross-modal understanding and generation.

### Multi Modal Projector

The multi-modal projector takes the vision tokens and projects them into the same embedding space as the language model. This is done through a simple linear transformation followed by an RMS normalization. The resulting projected embeddings can then be integrated with the text embeddings in the language model, allowing for rich cross-modal interactions between the visual and textual information.

In [62]:
def gemma3_mm_projector(
        vision_outputs: jnp.ndarray,   # (1, 256, d_vision=1152)
        params: dict,
        config: Gemma3Config
    ) -> jnp.ndarray:                  # (1, 256, hidden_size=2560)
    # vision_outputs is already (B, 256, d_vision) — no pooling needed
    normed = rms_norm(vision_outputs, params['norm'])
    projected = jnp.dot(normed, params['proj'])   # (1, 256, 2560)
    return projected

In [95]:
def fuse_multimodal_inputs(
        input_ids: jnp.ndarray,        # (seq_len,)
        text_embeddings: jnp.ndarray,  # (seq_len, hidden_size)
        image_features: jnp.ndarray,   # (1, 256, hidden_size)
        image_token_id: int
    ) -> tuple[jnp.ndarray, jnp.ndarray]:
    special_image_mask = (input_ids == image_token_id)
    inputs_embeds = text_embeddings.at[special_image_mask].set(image_features.reshape(-1, 2560))
    token_type_ids = jnp.where(special_image_mask, 1, 0)
    token_type_ids = token_type_ids[None, :]
    return inputs_embeds, token_type_ids

### Multimodal Causal Mask

In a standard Transformer, the causal mask is a simple lower-triangular matrix that prevents tokens from looking into the future. But for Gemma 3, we have to follow a different approach.

Because an image is a spatial object, its 256 tokens need to see each other bidirectionally (all at once) to understand context. If we used a standard causal mask on an image, the top-left pixel couldn't see the bottom-right pixel, and the model would lose the "big picture."

The Gemma 3 mask is a fusion of two behaviors:
- Text Tokens: Follow strict Causal logic (can only see the past).
- Image Tokens: Follow Block-Diagonal logic (can see all other tokens within the same 256-token image block).

In [49]:
def create_multimodal_causal_mask(
        token_type_ids: jnp.ndarray,  # (1, seq_len)
        max_seq_len: int,
        tokens_per_image: int = 256
    ) -> jnp.ndarray:                 # (1, seq_len, max_seq_len)

    batch_size, seq_len = token_type_ids.shape

    q_idx = jnp.arange(seq_len)      # (seq_len,)
    k_idx = jnp.arange(max_seq_len)  # (max_seq_len,)

    # Standard causal: query at position i can see keys 0..i
    causal_mask = k_idx[None, :] <= q_idx[:, None]  # (seq_len, max_seq_len)

    # Image group ids for query tokens
    is_image = (token_type_ids == 1)  # (1, seq_len)
    shifted_is_image = jnp.pad(is_image[:, :-1], ((0, 0), (1, 0)), constant_values=0)
    new_image_start = is_image & ~shifted_is_image
    image_group_ids = jnp.cumsum(new_image_start, axis=1) - 1  # (1, seq_len)
    image_group_ids = jnp.where(is_image, image_group_ids, -1)

    # Pad key group ids to max_seq_len; positions beyond seq_len get -1 (not image)
    key_group_ids = jnp.pad(
        image_group_ids, ((0, 0), (0, max_seq_len - seq_len)), constant_values=-1
    )  # (1, max_seq_len)

    q_groups = image_group_ids[:, :, None]  # (1, seq_len, 1)
    k_groups = key_group_ids[:, None, :]    # (1, 1, max_seq_len)

    bidirectional_allowed = (q_groups == k_groups) & (q_groups != -1)  # (1, seq_len, max_seq_len)
    final_mask = jnp.logical_or(causal_mask[None, :, :], bidirectional_allowed)

    # Binary 0/1 to match get_causal_mask format (attention block applies (1 - mask) * -1e10)
    return final_mask.astype(jnp.bfloat16)


Visualize the attention mask to confirm that it has the expected structure, with bidirectional attention within image token groups and causal masking elsewhere.

In [ ]:
import matplotlib.pyplot as plt

def plot_attention_mask(
        mask: jnp.ndarray,              # (seq_len, seq_len)
        title: str = "Gemma 3 Multimodal Mask"
    ) -> None:
    mask_to_plot = mask.astype(jnp.float32)
    plt.figure(figsize=(8, 8))
    plt.imshow(mask_to_plot, cmap='binary', interpolation='nearest')
    plt.title(title)
    plt.xlabel("Key Tokens (What can be seen)")
    plt.ylabel("Query Tokens (Who is looking)")
    plt.grid(False)
    plt.show()

## Gemma Multimodal

Now let's put everything together. Use the widget below to type a prompt and optionally provide one or more local image paths. When you're ready, hit **Submit** — the input is captured in a `captured` dict that the following cells will read from.


In [100]:
import ipywidgets as widgets
from IPython.display import display, clear_output

prompt_input = widgets.Textarea(
    value='',
    placeholder='Type your prompt here...',
    description='Prompt:',
    layout=widgets.Layout(width='100%', height='100px')
)

image_input = widgets.Text(
    value='',
    placeholder='Optional: comma-separated image paths, e.g. /path/img1.jpg, /path/img2.jpg',
    description='Images:',
    layout=widgets.Layout(width='100%')
)

submit_btn = widgets.Button(
    description='Submit',
    button_style='primary',
    layout=widgets.Layout(width='120px', margin='8px 0 0 0')
)

status_output = widgets.Output()

# Captured state accessible by subsequent cells
captured = {'prompt': None, 'image_paths': []}

def on_submit(b):
    with status_output:
        clear_output()
        prompt = prompt_input.value.strip()
        raw_paths = [p.strip() for p in image_input.value.split(',') if p.strip()]
        captured['prompt'] = prompt
        captured['image_paths'] = raw_paths
        n_imgs = len(raw_paths)
        print(f"Captured: prompt ({len(prompt)} chars)" + (f", {n_imgs} image path(s)" if n_imgs else ", no images"))

submit_btn.on_click(on_submit)

display(widgets.VBox([
    widgets.Label("Enter a text prompt and optionally one or more image paths, then click Submit."),
    prompt_input,
    image_input,
    submit_btn,
    status_output,
]))


`classify_and_load_input` reads from the `captured` dict, validates any image paths, and returns three things: the **mode** (`"text-only"` or `"multimodal"`), a clean **prompt string**, and a list of **preprocessed image arrays** ready for the vision tower.


> **Note:** This implementation places all image tokens before the text prompt.
> Interleaving images within text (e.g. one `<image>` per sentence) would require
> additional steps.

In [101]:
def classify_and_load_input(captured: dict):
    prompt = captured.get('prompt') or ''
    image_paths = captured.get('image_paths', [])

    if not prompt:
        raise ValueError("Prompt is empty. Fill in the prompt widget and click Submit first.")

    valid_images, missing = [], []
    for p in image_paths:
        (valid_images if os.path.isfile(p) else missing).append(p)

    if missing:
        print(f"Warning: {len(missing)} path(s) not found and will be skipped: {missing}")

    mode = 'multimodal' if valid_images else 'text-only'

    print(f"Input mode : {mode}")
    print(f"Prompt     : {repr(prompt[:120])}{'...' if len(prompt) > 120 else ''}")

    loaded_images = []
    if valid_images:
        for i, path in enumerate(valid_images):
            img = preprocess_image(path)
            loaded_images.append(img)
            print(f"Image {i+1}    : {path}  →  shape {img.shape}")

    return mode, prompt, loaded_images

input_mode, user_prompt, user_images = classify_and_load_input(captured)

Input mode : multimodal
Prompt     : 'Can you describe the images.'
Image 1    : cat.png  →  shape (896, 896, 3)
Image 2    : car.png  →  shape (896, 896, 3)


`prepare_multimodal_input` is the glue that ties the vision and language pipelines together. For each image it runs the full SigLIP → token condensation → MM projector pipeline, then builds a tokenized prompt with one `<image_soft_token>` block per image and calls `fuse_multimodal_inputs` to splice the projected image features into the right positions in the embedding sequence. The result is `input_embeds` and `token_type_ids`, which are all the language model needs to do a multimodal prefill.


Gemma 3 uses a specific vocabulary ID, typically 262144, which acts as a "landing strip." When you write a prompt like "(image) What is this?", the tokenizer produces a sequence where (image) is just one single token.
We must expand this single slot into 256 slots to fit our projected SigLIP tokens.

In [102]:
def prepare_multimodal_input(
        user_prompt: str,
        user_images: list[jnp.ndarray],   # list of (896, 896, 3) arrays
        m: dict,
        config: Gemma3Config,
        tokenizer,
        tokens_per_image: int = 256
    ) -> tuple[jnp.ndarray, jnp.ndarray]:

    image_encodings = jnp.concatenate(
        [siglip_transformer(img, m['vision_model']) for img in user_images],
        axis=0
    )  # (N, 256, 1152)
    image_features = gemma3_mm_projector(image_encodings, m['mm_projector'], config)

    image_slots = ("<image_soft_token>" * tokens_per_image) * len(user_images)
    full_prompt = "<start_of_turn>user\n" + image_slots + user_prompt + "<end_of_turn>"
    input_ids = tokenizer.encode(full_prompt)

    text_embeddings = m['language_model']['embed'][jnp.array(input_ids)] * jnp.sqrt(config.hidden_size)
    input_embeds, token_type_ids = fuse_multimodal_inputs(
        jnp.array(input_ids), text_embeddings, image_features, config.image_token_index
    )
    return jnp.array(input_ids), input_embeds, token_type_ids

In [103]:
input_ids, input_embeds, token_type_ids = prepare_multimodal_input(user_prompt, user_images, m, config, tokenizer)
input_ids.shape, input_embeds.shape, token_type_ids.shape

((523,), (523, 2560), (1, 523))

For a multimodal prompt we need the special block-diagonal causal mask we built earlier — image token blocks get bidirectional attention while text tokens stay strictly causal. For a text-only prompt we skip it and let the standard causal mask inside `gemma_text_model_prefill` take over.


In [104]:
if len(user_images) > 0:
    causal_mask = create_multimodal_causal_mask(token_type_ids, tokens_per_image=256, max_seq_len=8192)
else:
    causal_mask = None

Same two-phase generation loop as before — **prefill** processes the full fused sequence (text + image embeddings) in one shot, then **decode** generates new tokens one at a time, attending to the KV cache.


In [105]:
cache = [{
            'k': jnp.zeros((config.max_seq_len, config.num_kv_heads, config.head_dim), dtype=jnp.bfloat16),
            'v': jnp.zeros((config.max_seq_len, config.num_kv_heads, config.head_dim), dtype=jnp.bfloat16),
        } for _ in range(config.num_layers)]

start_pos = 0
key = jax.random.PRNGKey(42)
input_seq_len = input_embeds.shape[0]

print("Generating: ")

logits, cache = gemma_text_model_prefill(input_ids, m['language_model'], cache, rotary_embeddings_table, config, start_pos, input_embeds, causal_mask)
next_token = sample_token(logits[input_seq_len - 1], temperature=0.2, key=key)
all_tokens = list(input_ids[:input_seq_len]) + [int(next_token)]

print(user_prompt)
print(tokenizer.decode([int(next_token)]))

for step in range(100):
    start_pos = len(all_tokens) - 1
    logits, cache = gemma_text_model_generate(jnp.array([next_token]), m['language_model'], cache, rotary_embeddings_table, config, start_pos)
    next_token = sample_token(logits[-1], temperature=0.3, key=key)
    all_tokens.append(int(next_token))
    print(tokenizer.decode([int(next_token)]), end="", flush=True)

    if next_token == 106:
        break

Generating: 
Can you describe the images.




Here's a description of the images:

**Image 1: Cat**

*   **Subject:** A ginger tabby cat.
*   **Pose:** The cat is sitting upright, looking directly at the camera with a slightly quizzical expression.
*   **Setting:** The background is blurred, suggesting an outdoor environment.
*   **Color:** The cat is a vibrant orange-ginger color.

**Image 2: Truck**

*   **Subject:** A